# F00 环境、图表与基线纪律


## 这个基础 lab 是为了解决什么

这本 lab 用来消除最常见的小白阻塞项，再进入文章优先主线。


## 做完后你应该带走的技能

- 在本地和 Colab 两种环境里稳定运行 notebook。
- 区分 baseline 与 variant，而不是把一次改动写成一团。
- 看懂最基本的 loss、metric 和 seed 波动。


## 最后交付这些产物

- 1 份第一次可复查的 experiment log。
- 1 张 baseline 与 variant 的对比图。
- 1 段说明你到底改了什么。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(7)
features = torch.randn(256, 2)
true_weights = torch.tensor([1.8, -1.1])
logits = features @ true_weights + 0.25 * torch.randn(256)
labels = (logits > 0).float().unsqueeze(1)
train_x, val_x = features[:192], features[192:]
train_y, val_y = labels[:192], labels[192:]


def train_run(weight_decay: float, seed: int):
    torch.manual_seed(seed)
    model = nn.Linear(2, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.25, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    train_losses, val_losses = [], []
    for _ in range(60):
        optimizer.zero_grad()
        train_logits = model(train_x)
        train_loss = loss_fn(train_logits, train_y)
        train_loss.backward()
        optimizer.step()
        with torch.no_grad():
            val_logits = model(val_x)
            val_loss = loss_fn(val_logits, val_y)
        train_losses.append(float(train_loss.detach()))
        val_losses.append(float(val_loss.detach()))
    with torch.no_grad():
        val_pred = (torch.sigmoid(model(val_x)) > 0.5).float()
        val_acc = float((val_pred.eq(val_y)).float().mean())
    return train_losses, val_losses, val_acc


baseline = train_run(weight_decay=0.0, seed=11)
variant = train_run(weight_decay=0.08, seed=11)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(baseline[0], label="baseline train", color="#1f5f8b")
axes[0].plot(baseline[1], label="baseline val", color="#1f5f8b", linestyle="--")
axes[0].plot(variant[0], label="variant train", color="#c96a28")
axes[0].plot(variant[1], label="variant val", color="#c96a28", linestyle="--")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].bar(["baseline", "variant"], [baseline[2], variant[2]], color=["#1f5f8b", "#c96a28"])
axes[1].set_ylim(0.7, 1.0)
axes[1].set_title("Validation accuracy")
plt.tight_layout()

print("Baseline val accuracy:", round(baseline[2], 3))
print("Variant val accuracy:", round(variant[2], 3))
print("Judgment call: the variant changes regularization, so the baseline/variant pair is legible.")


## 小结

真正的研究起点，不是“能跑”，而是 baseline、variant、指标和 judgment call 都能在写作里分开。


## 验收题

- 为什么一个只会“跑通”的实验，还不能叫研究输入？
- 如果 baseline 没写清楚，后面的 sweep 和 ablation 会在哪些地方失真？
- 你会用什么最小日志字段，保证两天后还能复盘这次 run？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就先不要进入文章主线。
